# 深入理解：低通滤波器为何导致符号展宽？

符号展宽的本质是**信道的带宽受限**。在频域上，这表现为高频分量的衰减；在时域上，则表现为脉冲响应变宽，产生前导和拖尾。

本 Notebook 通过以下三组可视化来说明：
1. 低通滤波器的**幅频响应**及其**冲激响应**（时域）—— 展示滤波器自身的“展宽”特性。
2. 理想脉冲的**频谱**与经过滤波后的**频谱**对比 —— 展示高频被削减。
3. 时域卷积的结果 —— 直观看到脉冲被展宽，产生前导和拖尾。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.fft import fft, fftfreq, fftshift

# 设置中文字体（可选）
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 通用参数
fs = 1000               # 采样率 (Hz)
T = 2.0                 # 总时长 (s)
N = int(fs * T)         # 采样点数
t = np.linspace(0, T, N, endpoint=False)

## 1. 低通滤波器的频率响应与冲激响应

我们使用一个二阶巴特沃斯低通滤波器，截止频率设为 20 Hz。

In [ ]:
# 设计滤波器
fc = 20                         # 截止频率 (Hz)
b, a = signal.butter(2, fc/(fs/2), btype='low')

# 计算频率响应
w, h = signal.freqz(b, a, worN=8000, fs=fs)  # w 是频率数组，h 是复数响应

# 计算冲激响应（时域）
impulse = np.zeros(N)
impulse[0] = 1
impulse_response = signal.filtfilt(b, a, impulse)  # 实际上应该用 lfilter，但为了与之前一致
# 注意：filtfilt 是零相位滤波，会产生非因果的冲激响应。为了准确展示滤波器的因果冲激响应，应使用 lfilter
# 这里为了简单，使用 lfilter
impulse_response = signal.lfilter(b, a, impulse)

# 绘制
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 左图：幅频响应
ax1.plot(w, 20*np.log10(np.abs(h)), 'b')
ax1.axvline(fc, color='r', linestyle='--', label=f'截止频率 {fc} Hz')
ax1.set_title('低通滤波器幅频响应')
ax1.set_xlabel('频率 (Hz)')
ax1.set_ylabel('幅度 (dB)')
ax1.grid(True)
ax1.legend()

# 右图：冲激响应（时域）
ax2.plot(t[:200], impulse_response[:200], 'g')  # 只显示前200ms
ax2.set_title('低通滤波器冲激响应 (时域)')
ax2.set_xlabel('时间 (秒)')
ax2.set_ylabel('幅度')
ax2.grid(True)
ax2.set_xlim(0, 0.2)

plt.tight_layout()
plt.show()

**解读**：
- 左图显示滤波器对高频分量（>20Hz）有显著衰减，这正是“低通”的含义。
- 右图是滤波器的冲激响应，它本身就不是一个理想的冲激，而是具有一定的持续时间（展宽）。这意味着，当一个理想脉冲通过该系统时，输出就是该冲激响应，因此必然在时间上扩散。

## 2. 理想脉冲的频谱与滤波后的频谱

我们构造一个宽度为 0.1 秒的矩形脉冲，观察其频谱在滤波前后的变化。

In [ ]:
# 生成矩形脉冲
pulse_width = 0.1
pulse = np.zeros(N)
start_idx = int(0.5 * fs)
end_idx = int((0.5 + pulse_width) * fs)
pulse[start_idx:end_idx] = 1.0

# 通过滤波器
filtered_pulse = signal.lfilter(b, a, pulse)

# 计算频谱（使用FFT）
freqs = fftfreq(N, 1/fs)[:N//2]  # 只取正频率
pulse_spectrum = np.abs(fft(pulse))[:N//2]
filtered_spectrum = np.abs(fft(filtered_pulse))[:N//2]

# 绘制
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(freqs, pulse_spectrum, 'b', label='原始脉冲频谱')
plt.plot(freqs, filtered_spectrum, 'r', label='滤波后脉冲频谱')
plt.axvline(fc, color='k', linestyle='--', label=f'截止频率 {fc} Hz')
plt.title('脉冲频谱对比')
plt.xlabel('频率 (Hz)')
plt.ylabel('幅度')
plt.grid(True)
plt.legend()
plt.xlim(0, 200)

plt.subplot(1, 2, 2)
plt.plot(t, pulse, 'b', label='原始脉冲')
plt.plot(t, filtered_pulse, 'r', label='滤波后脉冲')
plt.title('时域波形对比')
plt.xlabel('时间 (秒)')
plt.ylabel('幅度')
plt.grid(True)
plt.legend()
plt.xlim(0.3, 0.9)

plt.tight_layout()
plt.show()

**解读**：
- 左图显示原始脉冲的频谱具有无限带宽（理论上，矩形脉冲的频谱是sinc函数，旁瓣无限延伸），而经过低通滤波后，高于20Hz的分量被急剧衰减。
- 右图是时域波形，红色波形（滤波后）比蓝色波形（原始）明显展宽，且出现了前导和拖尾。这正是因为高频成分的丢失导致边沿变缓，能量在时间轴上扩散。

## 3. 从卷积角度理解

时域上，滤波输出等于输入信号与滤波器冲激响应的卷积。我们将滤波器冲激响应与原始脉冲进行卷积，验证时域展宽。

In [ ]:
# 计算卷积（直接用之前滤波的结果即可，因为 lfilter 就是卷积的离散实现）
# 这里为了可视化，将冲激响应画得更长一些

# 截取冲激响应的主要部分（前200点）
imp_short = impulse_response[:200]
t_imp = t[:200]

# 绘制卷积示意图（动画效果静态）
plt.figure(figsize=(10, 6))

plt.subplot(3, 1, 1)
plt.plot(t, pulse, 'b')
plt.title('输入信号 x(t) — 理想矩形脉冲')
plt.xlim(0.3, 0.9)
plt.grid(True)

plt.subplot(3, 1, 2)
plt.plot(t_imp, imp_short, 'g')
plt.title('滤波器冲激响应 h(t)')
plt.xlabel('时间 (秒)')
plt.grid(True)

plt.subplot(3, 1, 3)
plt.plot(t, filtered_pulse, 'r')
plt.title('输出信号 y(t) = x(t) * h(t) — 展宽后的脉冲')
plt.xlabel('时间 (秒)')
plt.xlim(0.3, 0.9)
plt.grid(True)

plt.tight_layout()
plt.show()

## 4. 结论

- **频域视角**：低通滤波器切除了信号的高频分量，相当于在频域对信号的频谱进行加窗。由于信号的带宽变窄，其时域波形必然展宽（傅里叶变换的对偶性：频域越窄，时域越宽）。
- **时域视角**：滤波过程等价于信号与滤波器冲激响应的卷积。滤波器的冲激响应本身就具有一定的时间宽度，因此卷积后信号的持续时间增加，表现为前导和拖尾。
- **符号展宽的直接后果**：当传输连续比特时，每个比特的拖尾会叠加到相邻比特上，造成码间干扰（ISI）。

这就是低通滤波器导致符号展宽的物理本质。